# Day4 - Part3: LLM의 두뇌를 빌려 자율적으로 행동하는 AI, 에이전트(Agent) 구축하기

### 개요

Day 4의 두 번째 파트에서 우리는 RAG(검색 증강 생성)를 통해 LLM에게 '오픈북'을 쥐여주었습니다. 

이를 통해 LLM은 특정 지식 베이스를 참고하여 사실에 기반한 답변을 생성할 수 있게 되었죠. 

하지만 LLM의 잠재력은 단순히 정보를 검색하고 요약하는 데 그치지 않습니다. 

만약 LLM이 스스로 '생각'하고, '계획'을 세우고, 필요한 '도구'를 사용하여 문제를 해결할 수 있다면 어떨까요?

  * "오늘 서울 날씨 어때?" 라는 질문에 현재 날씨 정보를 웹에서 검색해서 알려준다.
  
  * "A 회사와 B 회사의 최근 3개월간 주가 추이를 비교해 줘." 라는 요청에 금융 API를 호출하여 데이터를 가져오고, 그 결과를 분석하여 그래프로 보여준다.
  * "내일 오전 10시에 'AI 팀 주간 회의' 일정을 캘린더에 추가해 줘." 라고 말하면, 실제 캘린더 API를 사용해 일정을 등록한다.

이처럼 LLM이 단순히 텍스트를 생성하는 것을 넘어, 외부 세계와 상호작용하며 주어진 목표를 달성하기 위해 자율적으로 행동하는 프로그램을 `AI 에이전트(Agent)` 라고 부릅니다. 

에이전트는 LLM을 '두뇌' 즉, 추론 엔진(Reasoning Engine)으로 사용하여 어떤 행동을 취할지 결정합니다.

 RAG가 LLM의 '지식'을 확장했다면, 에이전트는 LLM의 '행동'을 확장하는 기술입니다.

이번 파트에서는 LangChain 프레임워크를 사용하여 우리만의 AI 에이전트를 구축하는 방법을 배웁니다. 

에이전트의 핵심 구성 요소인 `도구(Tools)`, `프롬프트(Prompts)`, 그리고 이 모든 것을 조율하는 `에이전트 실행기(Agent Executor)` 의 개념을 이해하고, 실제 최신 뉴스 기사를 분석하는 RAG 시스템과 웹 검색 기능을 '도구'로 장착한 멀티-툴(Multi-tool) Q\&A 봇을 직접 만들어 보겠습니다.

`이번 파트의 학습 목표:`

  * 표준 LLM 및 RAG 시스템과 AI 에이전트의 차이점을 설명할 수 있습니다.
  
  * 에이전트의 핵심 동작 원리인 `ReAct (Reason, Act)` 프레임워크를 이해합니다.
  * 에이전트의 핵심 구성 요소인 `도구(Tool), LLM, 프롬프트(Prompt), 실행기(Executor)` 의 역할을 이해하고 코드로 구현할 수 있습니다.
  * `LangChain`을 사용하여 여러 도구(RAG 검색, 웹 검색 등)를 정의하고, 이를 활용하는 AI 에이전트를 구축할 수 있습니다.
  * 대화의 맥락을 기억하는 `메모리(Memory)`를 에이전트에 통합하여 자연스러운 셔봇을 구현할 수 있습니다.
  * 최신 뉴스 기사에 대한 심층 분석과 웹 검색이 모두 가능한 `멀티-툴 AI 에이전트`를 처음부터 끝까지 구현할 수 있습니다.



-----

### 1. AI 에이전트는 어떻게 작동하는가?: ReAct 프레임워크

> 논문명: ReAct: Synergizing Reasoning and Acting in Language Models
> [논문링크](https://arxiv.org/abs/2210.03629)

AI 에이전트의 마법은 LLM의 '추론' 능력에 기반합니다. LangChain 에이전트가 주로 사용하는 접근법 중 하나는 `ReAct (Reason + Act)` 프레임워크입니다. 

이름에서 알 수 있듯이, LLM은 `생각(Thought)` 하고, 그 생각에 따라 `행동(Action)` 하며, 그 행동의 `결과(Observation)` 를 다시 관찰하여 다음 생각을 이어가는 과정을 반복합니다.

이 과정을 통해 에이전트는 복잡한 문제를 여러 단계로 나누어 해결할 수 있습니다.

1.  `Thought (생각):` 사용자의 질문을 받고, 목표를 달성하기 위해 무엇을 해야 할지 생각합니다. '어떤 도구를 사용해야 할까?', '정보가 충분한가?', '다음 단계는 무엇인가?' 등을 LLM 스스로 판단합니다.

2.  `Action (행동):` 생각의 결과로, 특정 '도구'를 사용하기로 결정하고 실행합니다. 예를 들어, 'WebSearch' 도구를 '2025년 AI 트렌드'라는 입력값으로 호출합니다.
3.  `Observation (관찰):` 행동의 결과를 얻습니다. 'WebSearch' 도구는 검색 결과 텍스트를 반환하겠죠. 이것이 관찰 결과입니다.
4.  `Repeat (반복):` 에이전트는 이 관찰 결과(Observation)를 가지고 다시 `생각(Thought)` 단계로 돌아갑니다. '검색 결과를 보니 정보가 충분하군. 이제 답변을 생성해야겠다.' 또는 '정보가 부족하니 다른 키워드로 다시 검색해야겠다.' 와 같이 다음 행동을 결정합니다.
5.  `Final Answer (최종 답변):` 이 사이클을 반복하다가, 마침내 최종 답변을 생성할 수 있다고 판단되면, 사용자에게 답변을 반환하고 작업을 종료합니다.

이 `Thought -> Action -> Observation` 루프가 바로 에이전트가 자율적으로 문제를 해결하는 핵심 원리입니다.

-----


### 2. 에이전트의 핵심 구성 요소

LangChain으로 에이전트를 만들려면 몇 가지 핵심 요소를 조립해야 합니다. 각 요소가 어떤 역할을 하는지 코드를 통해 알아봅시다.

#### 2.1. LLM: 에이전트의 '두뇌'

모든 결정의 중심에는 LLM이 있습니다. 우리는 세계에서 가장 강력한 모델 중 하나인 OpenAI의 `gpt-4o`를 에이전트의 두뇌로 사용할 것입니다.

In [1]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv()

# .env 파일에 키가 제대로 설정되었는지 확인 (옵션)
# is_key_available = os.getenv("OPENAI_API_KEY") is not None
# print(f"OpenAI API Key is available: {is_key_available}")

True

In [2]:
# LLM 초기화
from langchain_openai import ChatOpenAI

# temperature는 모델의 창의성/일관성을 조절합니다. 0에 가까울수록 일관적인 답변을 생성합니다.
llm = ChatOpenAI(model="gpt-4o", temperature=0)

print("LLM (에이전트의 두뇌) 준비 완료!")

LLM (에이전트의 두뇌) 준비 완료!


#### 2.2. 도구 (Tools): 에이전트의 '손과 발'

도구는 에이전트가 외부 세계와 상호작용할 수 있게 해주는 함수입니다. 웹 검색, 계산기, 데이터베이스 조회 등 모든 기능이 도구가 될 수 있습니다. 에이전트에게 가장 중요한 것은 `각 도구의 설명(description)` 입니다. LLM은 이 설명을 읽고 언제 어떤 도구를 사용해야 할지 판단하기 때문입니다.

LangChain의 `@tool` 데코레이터를 사용하면 파이썬 함수를 매우 쉽게 도구로 만들 수 있습니다.

In [ ]:
# docstring을 필수로 작성해줘야 함
# 함수의 type을 엄격하게 정의해주는 게 좋음

In [3]:
from langchain.agents import tool
import datetime

@tool
def get_current_time(ignored_input: str = "") -> str: # type 지정정
    """
    현재 한국 시간을 'YYYY-MM-DD HH:MM:SS' 형식으로 반환합니다.
    """
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# 도구 리스트에 추가
tools = [get_current_time]

# 도구 정보 확인
print(f"등록된 도구: {[t.name for t in tools]}")
print(f"'{tools[0].name}' 도구 설명: {tools[0].description}")

등록된 도구: ['get_current_time']
'get_current_time' 도구 설명: 현재 한국 시간을 'YYYY-MM-DD HH:MM:SS' 형식으로 반환합니다.


#### 2.3. 프롬프트 (Prompts): 에이전트의 '행동 지침서'

에이전트는 내부적으로 매우 정교하게 설계된 프롬프트를 사용합니다. 이 프롬프트는 LLM에게 다음과 같은 정보를 제공합니다.

  * 너는 \~\~ 역할을 하는 에이전트다. (역할 부여)
  
  * 너는 다음과 같은 도구들을 사용할 수 있다. (도구 목록 및 설명)
  * `Thought -> Action -> Observation` 형식에 맞춰 생각하고 행동해야 한다. (ReAct 지침)
  * 사용자의 질문은 이것이다. (`input`)
  * 지금까지의 중간 생각과 행동의 기록은 이것이다. (`agent_scratchpad`)

다행히 LangChain은 이러한 복잡한 프롬프트를 자동으로 생성해주는 `ChatPromptTemplate.from_messages` 와 같은 기능을 제공하므로, 우리가 직접 모든 내용을 작성할 필요는 없습니다.

#### 2.4. 에이전트 실행기 (Agent Executor): 모든 것의 조율자

이제 두뇌(LLM), 손과 발(Tools), 그리고 행동 지침(Prompt)이 준비되었습니다. `

에이전트 실행기(Agent Executor)` 는 이 모든 것을 하나로 묶어 `Thought -> Action -> Observation` 루프를 실제로 실행하는 역할을 합니다.

LangChain의 최신 방식인 `create_openai_tools_agent` 와 `AgentExecutor`를 사용하여 에이전트를 조립하고 실행해 봅시다.

In [4]:
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 에이전트를 위한 프롬프트 템플릿을 정의합니다.
# 이 템플릿은 에이전트가 대화 기록(chat_history)과 중간 과정(agent_scratchpad)을
# 어떻게 활용할지 알려주는 중요한 역할을 합니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 주어진 도구를 활용하여 사용자의 질문에 답변하는 유능한 AI 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="chat_history", optional=True), # 대화 기록 (메모리)
    ("human", "{input}"), # 사용자의 입력
    MessagesPlaceholder(variable_name="agent_scratchpad"), # 에이전트의 중간 생각/행동 기록 - 저장을 통한 복기
])

# 1. LLM과 도구, 프롬프트를 연결하여 에이전트를 생성합니다.
agent = create_openai_tools_agent(llm, tools, prompt)

# 2. 생성된 에이전트와 도구들을 사용하여 에이전트 실행기를 만듭니다.
# verbose=True 옵션은 에이전트의 내부 추론 과정을 보여줍니다.
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 에이전트 실행
response = agent_executor.invoke({"input": "지금 몇 시야?"})

print("--- 최종 답변 ---")
print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `get_current_time` with `{}`


2025-06-26 16:46:09현재 시간은 2025년 6월 26일 16시 46분 9초입니다.

> Finished chain.
--- 최종 답변 ---
현재 시간은 2025년 6월 26일 16시 46분 9초입니다.


`verbose=True` 로 실행하면, 에이전트가 `get_current_time` 도구가 필요하다고 '생각'하고, '실행'하여 현재 시간 정보를 '관찰'한 뒤, 최종 답변을 만들어내는 전 과정을 엿볼 수 있습니다. 

이것이 바로 ReAct의 힘입니다.




-----

### 3. 종합 실습: 멀티-툴 AI 뉴스 분석가 만들기

이제 개념을 익혔으니, 더 실용적인 에이전트를 만들어 보겠습니다. 이 에이전트는 두 가지 강력한 도구를 가집니다.

1.  `RAG 검색 도구:` 우리가 제공한 최신 AI/데이터 관련 뉴스 기사 3건에 대한 내용을 질문하면, Supabase 벡터 DB에서 관련 내용을 찾아 답변하는 도구입니다.
2.  `웹 검색 도구:` RAG 문서에 없는 내용(예: "엔비디아의 최신 주가는?")을 질문하면, 실시간으로 웹을 검색하여 답변하는 도구입니다.

또한, 이전 대화 내용을 기억하는 `메모리(Memory)` 기능까지 탑재하여, 똑똑하고 자연스러운 대화가 가능한 AI 뉴스 분석가를 완성해 보겠습니다.

#### 3.1. 환경 설정 및 라이브러리 설치

프로젝트에 필요한 라이브러리를 설치합니다. `duckduckgo-search`는 웹 검색 도구를 위해, `supabase`는 벡터 DB를 위해, `langchainhub`는 에이전트 프롬프트를 쉽게 가져오기 위해 사용됩니다.

```bash
!pip install langchain openai langchain_openai langchain_community supabase python-dotenv beautifulsoup4 duckduckgo-search langchainhub -q
```

#### 3.2. API 키 및 Supabase 정보 설정

Part 2와 동일하게, `.env` 파일을 사용하여 OpenAI API 키와 Supabase 접속 정보를 안전하게 관리합니다.

```
# .env 파일 내용 예시
OPENAI_API_KEY="sk-..."
SUPABASE_URL="https://your-project-url.supabase.co"
SUPABASE_ANON_KEY="your-supabase-anon-key"
```

In [6]:
!pip install langchain openai langchain_openai langchain_community supabase python-dotenv beautifulsoup4 duckduckgo-search langchainhub -q

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

True

#### 3.3. 도구 준비 (Tool Preparation)

`도구 1: RAG 검색 도구 만들기 (Supabase 기반)`

먼저 Part 2에서 배운 RAG 파이프라인을 구축하여 '도구'로 만듭니다.

`1) 샘플 데이터 준비`

2025년 데이터/AI 분야의 최신 동향에 대한 가상 뉴스 기사를 준비합니다.

In [6]:
# 실습용 샘플 뉴스 데이터
news_articles = [
    {
        "title": "데이터브릭스, 차세대 'AI 거버넌스' 프레임워크 '유니티 카탈로그 2.0' 발표",
        "content": """
        2025년 6월 20일, 데이터 및 AI 기업 데이터브릭스는 연례 컨퍼런스에서 차세대 데이터 거버넌스 프레임워크인 '유니티 카탈로그 2.0'을 공개했다. 이 프레임워크는 데이터, AI 모델, 관련 파이프라인 전체에 걸쳐 통합된 보안 및 거버넌스를 제공하는 것이 특징이다. 특히, 생성형 AI 모델의 출력 결과에 대한 실시간 모니터링과 유해성 탐지 기능을 탑재하여 기업이 책임감 있는 AI를 구현할 수 있도록 지원한다. 데이터브릭스의 CEO는 "AI의 민주화는 강력한 거버넌스 위에서만 가능하다"며, "유니티 카탈로그 2.0은 데이터 팀과 보안 팀 사이의 간극을 메우는 핵심 솔루션이 될 것"이라고 강조했다.
        """,
        "source": "데이터 이코노미"
    },
    {
        "title": "AI 옵저버빌리티(Observability) 플랫폼 '뉴럴와처', 2억 달러 투자 유치",
        "content": """
        AI 모델의 운영 상태를 실시간으로 감시하고 분석하는 'AI 옵저버빌리티' 분야가 급성장하고 있다. 관련 스타트업 '뉴럴와처(NeuralWatcher)'는 최근 시리즈 C 펀딩에서 2억 달러 규모의 투자를 유치했다고 밝혔다. 뉴럴와처의 플랫폼은 LLM 애플리케이션에서 발생하는 데이터 드리프트, 성능 저하, 환각(Hallucination) 현상을 자동으로 탐지하고 개발자에게 경고한다. 이를 통해 기업은 AI 서비스의 신뢰도를 높이고 예상치 못한 오류로 인한 비즈니스 손실을 최소화할 수 있다. 업계 전문가들은 AI 모델이 복잡해질수록 옵저버빌리티의 중요성은 더욱 커질 것이라고 전망했다.
        """,
        "source": "AI 스타트업 위클리"
    },
    {
        "title": "구글 클라우드, '버텍스 AI 에이전트 빌더' 정식 출시… 노코드 AI 개발 시대 열어",
        "content": """
        구글 클라우드는 자사의 AI 플랫폼 버텍스 AI(Vertex AI)에 '에이전트 빌더' 기능을 정식으로 추가했다고 발표했다. 이 기능은 코딩 경험이 없는 사용자도 간단한 자연어 명령어와 그래픽 인터페이스를 통해 복잡한 AI 에이전트를 구축할 수 있도록 돕는다. 사용자는 지식 베이스(문서, 웹사이트 등)를 연결하고, 구글 검색, 지도, 캘린더 등 다양한 구글 서비스를 도구로 추가하여 고객 서비스 챗봇, 사내 업무 자동화 봇 등을 손쉽게 만들 수 있다. 이는 전문 개발자뿐만 아니라 현업 실무자들도 직접 AI 솔루션을 만들 수 있는 '노코드 AI' 시대의 본격적인 시작을 의미한다.
        """,
        "source": "클라우드 인사이트"
    }
]

# Document 객체로 변환
from langchain.docstore.document import Document
documents = [
    Document(
        page_content=article["content"],
        metadata={"title": article["title"], "source": article["source"]}
    ) for article in news_articles
]

`2) Supabase 벡터 스토어 설정 및 데이터 인덱싱`

문서를 분할하고 임베딩하여 Supabase에 저장합니다. 이 벡터 스토어가 RAG 검색의 기반이 됩니다.

In [7]:
# Supabase에서 agent_documents 테이블 생성 및 인덱스 설정
# 이 쿼리들을 Supabase SQL 편집기 또는 바로 아래 셀을 통해서 실행하세요.
drop_table_query = """
DROP TABLE IF EXISTS agent_documents;
"""

drop_index_query = """
DROP INDEX IF EXISTS agent_documents_embedding_idx;
"""

drop_function_query = """
DROP FUNCTION IF EXISTS match_agent_documents;
"""

# 1. 테이블 생성 (UUID 사용으로 변경)
create_table_query = """
CREATE TABLE IF NOT EXISTS agent_documents (
    id UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    content TEXT,
    metadata JSONB,
    embedding VECTOR(1536)
);
"""

# 2. 벡터 인덱스 생성 (pgvector 확장 필요)
create_index_query = """
CREATE INDEX IF NOT EXISTS agent_documents_embedding_idx 
ON agent_documents 
USING ivfflat (embedding vector_cosine_ops)
WITH (lists = 100);
"""

# 3. 벡터 유사도 검색 함수 생성 (UUID 타입으로 변경)
create_function_query = """
CREATE OR REPLACE FUNCTION match_agent_documents(
    query_embedding VECTOR(1536),
    match_count INTEGER DEFAULT 5,
    filter JSONB DEFAULT '{}'
)
RETURNS TABLE (
    id UUID,
    content TEXT,
    metadata JSONB,
    similarity FLOAT
)
LANGUAGE plpgsql
AS $$
BEGIN
    RETURN QUERY
    SELECT
        agent_documents.id,
        agent_documents.content,
        agent_documents.metadata,
        1 - (agent_documents.embedding <=> query_embedding) AS similarity
    FROM agent_documents
    WHERE agent_documents.metadata @> filter
    ORDER BY agent_documents.embedding <=> query_embedding
    LIMIT match_count;
END;
$$;
"""

print("Supabase SQL 쿼리 준비 완료!")


Supabase SQL 쿼리 준비 완료!


In [5]:
!pip install psycopg2

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 19.2 MB/s eta 0:00:00


In [8]:
import sqlalchemy
from sqlalchemy import text
# 환경 변수 로드

# Supabase 설정 (SB_ 환경변수 사용)
sb_host = os.getenv("SB_HOST")
sb_user = os.getenv("SB_USER") 
sb_password = os.getenv("SB_PASSWORD")
sb_db = os.getenv("SB_DB")
sb_port = os.getenv("SB_PORT")

# SQLAlchemy 연결 문자열 생성
database_url = f"postgresql://{sb_user}:{sb_password}@{sb_host}:{sb_port}/{sb_db}"

# SQLAlchemy 엔진 생성
engine = sqlalchemy.create_engine(database_url)

In [9]:
# 쿼리들을 순서대로 실행
queries = [
    drop_table_query,
    drop_index_query,
    drop_function_query,
    create_table_query,
    create_index_query, 
    create_function_query
]

try:
    with engine.connect() as connection:
        for i, query in enumerate(queries, 1):
            print(f"쿼리 {i} 실행 중...")
            connection.execute(text(query))
            connection.commit()
            print(f"쿼리 {i} 실행 완료!")
    
    print("모든 SQL 쿼리가 성공적으로 실행되었습니다!")
    
except Exception as e:
    print(f"SQL 실행 중 오류 발생: {e}")
    print("Supabase SQL 편집기에서 수동으로 실행해주세요.")


쿼리 1 실행 중...
쿼리 1 실행 완료!
쿼리 2 실행 중...
쿼리 2 실행 완료!
쿼리 3 실행 중...
쿼리 3 실행 완료!
쿼리 4 실행 중...
쿼리 4 실행 완료!
쿼리 5 실행 중...
쿼리 5 실행 완료!
쿼리 6 실행 중...
쿼리 6 실행 완료!
모든 SQL 쿼리가 성공적으로 실행되었습니다!


In [10]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores.supabase import SupabaseVectorStore
from supabase.client import create_client

# 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
split_documents = text_splitter.split_documents(documents)

# Supabase 클라이언트 및 임베딩 모델 초기화
supabase_url = os.getenv("SUPABASE_URL")
supabase_key = os.getenv("SUPABASE_ANON_KEY")
supabase = create_client(supabase_url, supabase_key)
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Supabase에 데이터 저장 및 인덱싱
# (주의: 최초 실행 시 테이블 생성 및 데이터 저장으로 시간이 소요될 수 있습니다.)
# (테이블 이름은 기존과 충돌하지 않도록 'agent_documents'로 지정합니다.)
vector_store = SupabaseVectorStore.from_documents(
    documents=split_documents,
    embedding=embedding_model,
    client=supabase,
    table_name="agent_documents",
    query_name="match_agent_documents"
)

# 리트리버 생성
retriever = vector_store.as_retriever()

`3) RAG 리트리버를 '도구'로 변환`

LangChain의 `create_retriever_tool` 함수를 사용하여 방금 만든 리트리버를 에이전트가 사용할 수 있는 도구로 포장합니다.

In [ ]:
# db 검색 툴

In [11]:
from langchain.tools import Tool

# 리트리버를 도구로 생성
# 도구의 이름과 설명을 명확하게 작성하는 것이 매우 중요합니다.
retriever_tool = Tool(
    name="news_document_search",
    description="2025년 최신 AI 및 데이터 관련 뉴스 기사에 대한 정보를 검색합니다. '유니티 카탈로그', 'AI 옵저버빌리티', '버텍스 AI 에이전트 빌더'와 관련된 질문에 유용합니다.",
    func=retriever.get_relevant_documents
)

`도구 2: 웹 검색 도구 만들기`

`DuckDuckGoSearchRun`을 사용하여 웹 검색 도구를 간단하게 추가합니다.

In [12]:
pip install -U duckduckgo-search

Note: you may need to restart the kernel to use updated packages.


In [13]:
from langchain_community.tools.ddg_search import DuckDuckGoSearchRun

web_search_tool = DuckDuckGoSearchRun()

# 웹 검색 도구 정보 확인
print(f"도구 이름: {web_search_tool.name}")
print(f"도구 설명: {web_search_tool.description}")

도구 이름: duckduckgo_search
도구 설명: A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.


#### 3.4. 메모리 및 최종 에이전트 생성

이제 두 개의 도구(`retriever_tool`, `web_search_tool`)가 준비되었습니다. 대화 기록을 저장할 메모리와 함께 최종 에이전트를 조립해 봅시다.

In [14]:
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationBufferWindowMemory

# 사용할 도구들을 리스트로 묶기
tools = [retriever_tool, web_search_tool]

# 에이전트 프롬프트 설정 (langchainhub에서 검증된 프롬프트 사용)
# hwchase17/openai-functions-agent는 OpenAI의 함수 호출 기능에 최적화된
# 에이전트 프롬프트로, 메모리와 함께 사용하기 좋습니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 유능한 AI 뉴스 분석가입니다. 주어진 뉴스를 분석하거나 웹 검색을 통해 사용자의 질문에 답변합니다."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])


# 대화 기록을 저장할 메모리 설정 (최근 5개의 대화 턴을 기억)
memory = ConversationBufferWindowMemory(
    k=5,
    memory_key="chat_history",
    return_messages=True
)

# LLM, 도구, 프롬프트를 연결하여 에이전트 생성
agent = create_openai_tools_agent(llm, tools, prompt)

# 에이전트 실행기 생성
# handle_parsing_errors=True는 에이전트가 LLM의 출력 파싱에 실패했을 때
# 사용자에게 오류를 안내하고 계속 작동하도록 돕는 안정장치입니다.
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory, # 메모리 추가
    verbose=True,
    handle_parsing_errors=True
)

print("멀티-툴 AI 뉴스 분석가 에이전트가 준비되었습니다.")

멀티-툴 AI 뉴스 분석가 에이전트가 준비되었습니다.


C:\Users\Admin\AppData\Local\Temp\ipykernel_20668\3092526118.py:20: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(


#### 3.5. AI 뉴스 분석가와 대화하기

이제 모든 준비가 끝났습니다. 에이전트가 상황에 맞게 적절한 도구를 선택하고, 대화의 맥락을 기억하는지 테스트해 봅시다.

In [ ]:
# news_document_search : db 검색 툴
# 2개의 툴 중에 판단해서 action

In [15]:
# 질문 1: RAG 도구를 사용해야 하는 질문
question1 = "데이터브릭스가 발표한 유니티 카탈로그 2.0의 주요 특징이 뭐야?"
response1 = agent_executor.invoke({"input": question1})
print(f"🤖 AI 분석가: {response1['output']}")



> Entering new AgentExecutor chain...

Invoking: `news_document_search` with `유니티 카탈로그 2.0`


[Document(metadata={'title': "데이터브릭스, 차세대 'AI 거버넌스' 프레임워크 '유니티 카탈로그 2.0' 발표", 'source': '데이터 이코노미'}, page_content='2025년 6월 20일, 데이터 및 AI 기업 데이터브릭스는 연례 컨퍼런스에서 차세대 데이터 거버넌스 프레임워크인 \'유니티 카탈로그 2.0\'을 공개했다. 이 프레임워크는 데이터, AI 모델, 관련 파이프라인 전체에 걸쳐 통합된 보안 및 거버넌스를 제공하는 것이 특징이다. 특히, 생성형 AI 모델의 출력 결과에 대한 실시간 모니터링과 유해성 탐지 기능을 탑재하여 기업이 책임감 있는 AI를 구현할 수 있도록 지원한다. 데이터브릭스의 CEO는 "AI의 민주화는 강력한 거버넌스 위에서만 가능하다"며, "유니티 카탈로그 2.0은 데이터 팀과 보안 팀 사이의 간극을 메우는 핵심 솔루션이 될 것"이라고 강조했다.'), Document(metadata={'title': "AI 옵저버빌리티(Observability) 플랫폼 '뉴럴와처', 2억 달러 투자 유치", 'source': 'AI 스타트업 위클리'}, page_content="AI 모델의 운영 상태를 실시간으로 감시하고 분석하는 'AI 옵저버빌리티' 분야가 급성장하고 있다. 관련 스타트업 '뉴럴와처(NeuralWatcher)'는 최근 시리즈 C 펀딩에서 2억 달러 규모의 투자를 유치했다고 밝혔다. 뉴럴와처의 플랫폼은 LLM 애플리케이션에서 발생하는 데이터 드리프트, 성능 저하, 환각(Hallucination) 현상을 자동으로 탐지하고 개발자에게 경고한다. 이를 통해 기업은 AI 서비스의 신뢰도를 높이고 예상치 못한 오류로 인한 비즈니스 손실을 최소화할 수 있다. 업계 전문가들은 AI 모델이 복잡해질수록 옵저버빌리티의 중요성은 더욱 커질 것이라고 

In [16]:
# 질문 2: 웹 검색 도구를 사용해야 하는 질문
question2 = "구글의 모회사인 알파벳의 현재 주가는 얼마야?"
response2 = agent_executor.invoke({"input": question2})
print(f"🤖 AI 분석가: {response2['output']}")



> Entering new AgentExecutor chain...

Invoking: `duckduckgo_search` with `{'query': 'Alphabet stock price'}`


A detailed overview of Alphabet Inc. (GOOG) stock, including real-time price, chart, key statistics, news, and more. See the latest Alphabet Inc Class A stock price (GOOGL:XNAS), related news, valuation, dividends and more to help you make your investing decisions. Track ALPHABET INC. (GOOG) price, historical values, financial information, price forecast, and insights to empower your investing journey | MSN Money A high-level overview of Alphabet Inc. (GOOG) stock. View (GOOG) real-time stock price, chart, news, analysis, analyst reviews and more. Get Alphabet Inc (GOOG.C) real-time stock quotes, news, price and financial information from Reuters to inform your trading and investments현재 알파벳(Alphabet Inc.)의 주가는 실시간으로 변동하므로, 정확한 가격을 확인하려면 금융 뉴스 웹사이트나 주식 거래 플랫폼에서 실시간 주가 정보를 확인하는 것이 좋습니다. 구체적인 주가 정보를 원하시면, 구글 파이낸스, 야후 파이낸스, 또는 로이터와 같은 사이트를 방문해 보세요.

> Finished chain.
🤖 AI 분석가: 

In [17]:
# 질문 3: 이전 대화 내용을 기억해야 하는 질문
question3 = "방금 내가 첫 번째로 물어봤던 기술을 만든 회사가 어디였지?"
response3 = agent_executor.invoke({"input": question3})
print(f"🤖 AI 분석가: {response3['output']}")



> Entering new AgentExecutor chain...
첫 번째로 물어보신 "유니티 카탈로그 2.0"은 데이터브릭스(Databricks)에서 발표한 기술입니다. 데이터브릭스는 데이터 및 AI 플랫폼을 제공하는 회사로, 유니티 카탈로그를 통해 데이터 관리 및 거버넌스를 강화하고 있습니다.

> Finished chain.
🤖 AI 분석가: 첫 번째로 물어보신 "유니티 카탈로그 2.0"은 데이터브릭스(Databricks)에서 발표한 기술입니다. 데이터브릭스는 데이터 및 AI 플랫폼을 제공하는 회사로, 유니티 카탈로그를 통해 데이터 관리 및 거버넌스를 강화하고 있습니다.


`verbose=True` 출력 결과를 보면, 첫 번째 질문에는 `news_document_search` 도구를, 두 번째 질문에는 `duckduckgo_search` 도구를 호출하는 것을 확인할 수 있습니다. 

세 번째 질문에는 별도의 도구 없이, `memory`에 저장된 대화 기록을 바탕으로 답변을 생성합니다.

이것으로 우리는 LLM의 추론 능력과 외부 도구, 그리고 메모리를 결합하여 단순한 챗봇을 넘어선, 진정한 의미의 `AI 에이전트`를 성공적으로 구축했습니다.